# Data Preprocessing for Peptide-HLA Binding Prediction

This notebook prepares the raw MHC ligand dataset for downstream model training.
The process involves:
1. Data extraction & cleaning
2. Resolving conflicts (Deduplication)
3. Filtering invalid sequences
4. Balancing the dataset with synthetic negatives
5. Saving to a processed format

We begin by installing specific versions of our required libraries to guarantee reproducibility.

In [1]:
%%capture
# Install strict versions for reproducibility
%pip install --index-url https://pypi.org/simple pandas==3.0.0 numpy==2.4.2 scipy==1.17.0 matplotlib==3.10.8 seaborn==0.13.2

In [2]:
import os
import types
import random
from typing import Any
from collections.abc import Sequence, Set, Mapping

import numpy as np
import pandas as pd
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns

class Globals:
    """
    Container for explicit global state.
    """
    RAW_DATA_PATH: str = 'data/raw/mhc_ligand_table_export.csv'
    PROCESSED_DATA_DIR: str = 'data/processed'
    PROCESSED_DATA_PATH: str = os.path.join(PROCESSED_DATA_DIR, 'training_data.json')
    # We will hold our working dataframes here instead of polluting true globals
    uncleaned_raw_df: pd.DataFrame | None = None
    raw_data_df: pd.DataFrame | None = None
    deduplicated_df: pd.DataFrame | None = None
    filtered_df: pd.DataFrame | None = None
    balanced_df: pd.DataFrame | None = None

# Ensure the processed data directory exists
os.makedirs(Globals.PROCESSED_DATA_DIR, exist_ok=True)


## 1. Data Extraction & Cleaning

We begin by loading the raw CSV file. We only keep columns relevant to the actual training: the peptide Sequence, the Qualitative measurement, and the Quantitative measurement.
 
We then rename these columns for easier programmatic access.

In [3]:
def load_raw_data(file_path: str) -> pd.DataFrame:
    """
    Loads the raw CSV data.
    """
    return pd.read_csv(file_path, low_memory=False)

def explore_raw_data(input_df: pd.DataFrame) -> None:
    """
    Prints information about the raw dataframe, including column types and a sample.
    """
    print(f"Raw dataset shape: {input_df.shape}")
    print("\nColumn types:")
    print(input_df.dtypes)
    print("\nSample of the raw data (first 3 rows):")
    
    try:
        from IPython.display import display
        display(input_df.head(3))
    except ImportError:
        print(input_df.head(3))

def clean_data(input_df: pd.DataFrame) -> pd.DataFrame:
    """
    Selects the relevant columns and renames them appropriately.
    """
    subset_df: pd.DataFrame = input_df[['Epitope - Name', 'Assay - Qualitative Measurement', 'Assay - Quantitative measurement']]
    cleaned_df: pd.DataFrame = subset_df.iloc[1:].copy()

    cleaned_df.rename(columns={
        'Epitope - Name': 'Sequence',
        'Assay - Qualitative Measurement': 'Qualitative measurement',
        'Assay - Quantitative measurement': 'Quantitative measurement'
    }, inplace=True)

    return cleaned_df

def summarize_data(input_df: pd.DataFrame) -> None:
    """
    Prints a summary of the dataset including sequence uniqueness and top-5 duplicates.
    """
    is_unique: bool = input_df['Sequence'].is_unique
    if is_unique:
        print("All sequences are unique.")
    else:
        total_names: int = len(input_df['Sequence'])
        unique_names_count: int = input_df['Sequence'].nunique()
        duplicate_names_count: int = total_names - unique_names_count

        perc_unique: float = (unique_names_count / total_names) * 100
        perc_duplicates: float = (duplicate_names_count / total_names) * 100

        print("\nSequences are NOT unique. Printing relevant statistics:")
        print(f"Total number of names: {total_names}")
        print(f"Number of unique names: {unique_names_count} ({perc_unique:.2f}%)")
        print(f"Number of duplicate entries: {duplicate_names_count} ({perc_duplicates:.2f}%)")

        print("\nTop-5 duplicates:")
        top_duplicates: pd.Series = input_df['Sequence'].value_counts().head(5)
        for seq, count in top_duplicates.items():
            print(f"  {seq}: {count} occurrences")

Globals.uncleaned_raw_df = load_raw_data(Globals.RAW_DATA_PATH)
explore_raw_data(Globals.uncleaned_raw_df)

Globals.raw_data_df = clean_data(Globals.uncleaned_raw_df)
summarize_data(Globals.raw_data_df)


Raw dataset shape: (48865, 4)

Column types:
Assay ID - IEDB IRI                     str
Epitope - Name                          str
Assay - Qualitative Measurement         str
Assay - Quantitative measurement    float64
dtype: object

Sample of the raw data (first 3 rows):


,Assay ID - IEDB IRI,Epitope - Name,Assay - Qualitative Measurement,Assay - Quantitative measurement
0,http://www.iedb.org/assay/2621627,ARFDSKDEILL,Positive,NaN
1,http://www.iedb.org/assay/3205017,RRFHFSQSL,Positive,NaN
2,http://www.iedb.org/assay/3203470,GRLLQQLAM,Positive,NaN



Sequences are NOT unique. Printing relevant statistics:
Total number of names: 48864
Number of unique names: 22885 (46.83%)
Number of duplicate entries: 25979 (53.17%)

Top-5 duplicates:
  GRILSGVVTK: 43 occurrences
  GRIGVITNR: 38 occurrences
  HRAQVIYTR: 37 occurrences
  GRLDAVLQR: 35 occurrences
  GRAPISNPGM: 34 occurrences


## 2. Filtering Invalid Sequences

Protein sequences consist of specific amino acids. We must filter out any sequences containing unexpected or invalid characters that represent modifications or artifacts not supported by our language model.

In [18]:
def filter_valid_sequences(input_df: pd.DataFrame) -> pd.DataFrame:
    """
    Removes rows where the standard 20 amino acid character set is violated.
    """
    initial_rows: int = len(input_df)
    # Standard 20 amino acids
    allowed_chars: frozenset = frozenset('ACDEFGHIKLMNPQRSTVWY')

    def contains_only_allowed_chars(sequence: str) -> bool:
        return all(char in allowed_chars for char in str(sequence))

    filtered_df: pd.DataFrame = input_df[input_df['Sequence'].apply(contains_only_allowed_chars)]
    
    removed_rows: int = initial_rows - len(filtered_df)
    remaining_rows: int = len(filtered_df)
    percentage_removed: float = (removed_rows / initial_rows) * 100 if initial_rows > 0 else 0.0
    print(f"Remaining rows: {remaining_rows}, initial rows: {initial_rows}. "
          f"Filtered out {removed_rows} rows ({percentage_removed:.2f}%) with invalid characters.")
    return filtered_df

Globals.filtered_df = filter_valid_sequences(Globals.raw_data_df)


Remaining rows: 48043, initial rows: 48864. Filtered out 821 rows (1.68%) with invalid characters.


## 3. Quantitative vs Qualitative Measurement Analysis

Before making decisions on how to resolve conflicting duplicates, we explore the dataset to understand the presence and alignment of our measurements. Specifically, we want to see:
1. How often sequences have both quantitative and qualitative data.
2. When both are present, how strongly our binary quantitative classification (< 500 nM = Positive) aligns with the subjective qualitative labels.

In [5]:
def analyze_measurement_presence(input_df: pd.DataFrame) -> None:
    has_qual = input_df['Qualitative measurement'].notna()
    has_quant = input_df['Quantitative measurement'].notna()
    total = len(input_df)
    
    quant_na_qual_na = (~has_quant & ~has_qual).sum()
    quant_na_qual_set = (~has_quant & has_qual).sum()
    quant_set_qual_na = (has_quant & ~has_qual).sum()
    quant_set_qual_set = (has_quant & has_qual).sum()
    
    print("Comparison of Quantitative and Qualitative Measurements (Percentages):\n")
    print("-" * 71)
    print(f"{'' :<20} | {'Qualitative NaN':<20} | {'Qualitative Set':<20}")
    print("-" * 21 + "|" + "-" * 22 + "|" + "-" * 21)
    
    def fmt_perc(cnt):
        return f"{(cnt/total)*100:.2f} %"
    
    print(f"{'Quantitative NaN':<20} | {fmt_perc(quant_na_qual_na):<20} | {fmt_perc(quant_na_qual_set):<20}")
    print(f"{'Quantitative Set':<20} | {fmt_perc(quant_set_qual_na):<20} | {fmt_perc(quant_set_qual_set):<20}")
    print("-" * 71)

print("Running measurement presence analysis...")
analyze_measurement_presence(Globals.filtered_df)


Running measurement presence analysis...
Comparison of Quantitative and Qualitative Measurements (Percentages):

-----------------------------------------------------------------------
                     | Qualitative NaN      | Qualitative Set     
---------------------|----------------------|---------------------
Quantitative NaN     | 0.00 %               | 95.40 %             
Quantitative Set     | 0.00 %               | 4.60 %              
-----------------------------------------------------------------------


**Conclusions on Presence:**
The vast majority of the data points (~95%) only provide a qualitative measurement, while ~5% provide both quantitative and qualitative measurements. We do not have examples that only supply quantitative readings without a qualitative label.

### 3.1 Alignment Analysis

For the subset of data (~5%) with both measurements, we want to verify how well the continuous numeric data matches the subjective labels. We map the quantitative data to a binary outcome (`< 500 nM` = Positive) and evaluate its alignment with the qualitative labels under two criteria:
1. **Pure Alignment**: Exact textual matching.
2. **Soft Alignment**: Binned binary matching where variations of 'Positive' (e.g., 'Positive-High') are treated equally.

In [6]:
# Map describing how we flatten diverse qualitative tags into a binary signal.
QUALITATIVE_MAPPING: Mapping = types.MappingProxyType({
    'Negative': 0,
    'Positive': 1,
    'Positive-High': 1,
    'Positive-Intermediate': 1,
    'Positive-Low': 1
})

def analyze_measurement_alignment(input_df: pd.DataFrame) -> None:
    has_qual = input_df['Qualitative measurement'].notna()
    has_quant = input_df['Quantitative measurement'].notna()
    both_df = input_df[has_qual & has_quant].copy()
    both_present = len(both_df)
    
    if both_present == 0:
        print("No rows with both measurements to align.")
        return
        
    def quant_to_qual_str(val):
        return 'Positive' if float(val) < 500 else 'Negative'
        
    both_df['Predicted_Qual_Str'] = both_df['Quantitative measurement'].apply(quant_to_qual_str)
    
    print("======================================================")
    print(f"Alignment Analysis (Predicted from Quantitative vs. Actual Qualitative)")
    print(f"Total rows with both measurements for analysis: {both_present}")
    print("======================================================\n")
    
    pure_aligned = (both_df['Predicted_Qual_Str'] == both_df['Qualitative measurement']).sum()
    print("--- Pure Alignment (Exact String Match) ---")
    print(f"Number of purely aligned rows: {pure_aligned} ({(pure_aligned/both_present)*100:.2f}%)\n")
    
    print("Pure Alignment Combinations (Counts / Percentages):")
    pure_crosstab = pd.crosstab(both_df['Predicted_Qual_Str'], both_df['Qualitative measurement'])
    
    pure_formatted = pure_crosstab.copy().astype(str)
    for col in pure_crosstab.columns:
        for row_idx in pure_crosstab.index:
            count = pure_crosstab.loc[row_idx, col]
            perc = (count / both_present) * 100
            pure_formatted.loc[row_idx, col] = f"{count} ({perc:.2f}%)"
            
    pure_formatted = pure_formatted.rename_axis('Predicted Qualitative', axis=0).rename_axis('Qualitative measurement', axis=1)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print(pure_formatted.to_string())
    print("\n")
    
    predicted_from_quant = (both_df['Quantitative measurement'].astype(float) < 500).astype(int)
    both_df['Predicted_Binary'] = predicted_from_quant
    
    actual_mapped = both_df['Qualitative measurement'].map(QUALITATIVE_MAPPING)
    both_df['Actual_Mapped'] = actual_mapped
    
    soft_aligned = (both_df['Predicted_Binary'] == both_df['Actual_Mapped']).sum()
    print("--- Soft Alignment (Using QUALITATIVE_MAPPING) ---")
    print(f"Number of softly aligned rows: {soft_aligned} ({(soft_aligned/both_present)*100:.2f}%)\n")
    
    print("Soft Alignment Combinations (Counts / Percentages):")
    soft_crosstab = pd.crosstab(both_df['Predicted_Binary'], both_df['Actual_Mapped'])
    
    soft_formatted = soft_crosstab.copy().astype(str)
    for col in soft_crosstab.columns:
        for row_idx in soft_crosstab.index:
            count = soft_crosstab.loc[row_idx, col]
            perc = (count / both_present) * 100
            soft_formatted.loc[row_idx, col] = f"{count} ({perc:.2f}%)"
            
    soft_formatted = soft_formatted.rename_axis('Predicted_Mapped', axis=0).rename_axis('Actual_Mapped', axis=1)
    print(soft_formatted.to_string())

analyze_measurement_alignment(Globals.filtered_df)


Alignment Analysis (Predicted from Quantitative vs. Actual Qualitative)
Total rows with both measurements for analysis: 2210

--- Pure Alignment (Exact String Match) ---
Number of purely aligned rows: 1463 (66.20%)

Pure Alignment Combinations (Counts / Percentages):
Qualitative measurement       Negative    Positive Positive-High Positive-Intermediate  Positive-Low
Predicted Qualitative                                                                               
Negative                 1378 (62.35%)  41 (1.86%)    57 (2.58%)            14 (0.63%)  277 (12.53%)
Positive                     0 (0.00%)  85 (3.85%)   170 (7.69%)           185 (8.37%)     3 (0.14%)


--- Soft Alignment (Using QUALITATIVE_MAPPING) ---
Number of softly aligned rows: 1821 (82.40%)

Soft Alignment Combinations (Counts / Percentages):
Actual_Mapped                 0             1
Predicted_Mapped                             
0                 1378 (62.35%)  389 (17.60%)
1                     0 (0.00%)  443 (2

**Conclusions on Alignment:**
From the alignment analysis above, we can see that when both qualitative and quantitative values are present, they are highly aligned (~82%). Because of this consistency, and recognizing that quantitative values (geometric mean < 500 nM) provide a more precise, continuous physical ground truth compared to subjective qualitative bucketing, our deduplication strategy will prioritize **Quantitative measurements**.

## 4. Resolving Conflicts (Deduplication)

Since our raw dataset contains duplicate sequences with potentially conflicting measurements, we must deduplicate the data. 
We implement a consensus strategy:
1.  **Quantitative priority**: If a sequence has quantitative values, we compute its geometric mean. A geometric mean < 500 nM implies binding (Positive).
2.  **Qualitative fallback**: For sequences lacking quantitative data, we use qualitative consistency. 

Note: A single sequence might have received qualitative labels like 'Positive-High', 'Positive-Intermediate', etc. We treat all variants of positive as simply 1 (Positive), and 'Negative' as 0. This unifies our task into a simple binary classification problem.

In [7]:
def calculate_quantitative_consensus(group_quantitatives: pd.Series) -> int:
    """
    Decides the binding status based on the geometric mean of quantitative measurements.
    """
    values: pd.Series = group_quantitatives.dropna().astype(float)
    geo_mean: float = gmean(values)
    return int(geo_mean < 500)

def process_quantitative_split(quant_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Handle deduplication for sequences with quantitative data.
    Returns:
        consensus_df: DataFrame with deduplicated sequences and their binary isPositive label
        stats_df: DataFrame containing the final consensus label joined with the original qualitative mapped labels
                  to help compute the alignment matrix printouts.
    """
    quant_consensus_results: list[dict[str, Any]] = []
    
    # We also want to track how this merged quantitative label compares to the original qualitative ones
    grouped_quant: pd.core.groupby.generic.DataFrameGroupBy = quant_df.groupby('Sequence')

    for name, group in grouped_quant:
        is_positive: int = calculate_quantitative_consensus(group['Quantitative measurement'])
        quant_consensus_results.append({
            'Sequence': name,
            'isPositive': float(is_positive)
        })
        
    consensus_df = pd.DataFrame(quant_consensus_results)
    
    # Track alignment stats for reporting: Merge the consensus back with the original df
    stats_df = pd.merge(quant_df[['Sequence', 'Qualitative_Mapped']].dropna(), 
                        consensus_df, 
                        on='Sequence', 
                        how='inner')
                        
    return consensus_df, stats_df

def process_qualitative_split(qual_only_df: pd.DataFrame) -> tuple[pd.DataFrame, int, int, int]:
    """
    Handle deduplication for sequences missing quantitative data.
    Returns the consensus dataframe, and the stats (consistent positive, consistent negative, conflicting).
    """
    qual_consensus_results: list[dict[str, Any]] = []
    
    consistent_pos = 0
    consistent_neg = 0
    conflicting = 0
    
    grouped_qual: pd.core.groupby.generic.DataFrameGroupBy = qual_only_df.groupby('Sequence')

    for name, group in grouped_qual:
        unique_values: np.ndarray = group['Qualitative_Mapped'].dropna().unique()
        
        if len(unique_values) == 1:
            val = float(unique_values[0])
            qual_consensus_results.append({
                'Sequence': name,
                'isPositive': val
            })
            if val == 1.0:
                consistent_pos += 1
            else:
                consistent_neg += 1
        elif len(unique_values) > 1:
            conflicting += 1
            
    return pd.DataFrame(qual_consensus_results), consistent_pos, consistent_neg, conflicting

def deduplicate_dataset(input_df: pd.DataFrame) -> pd.DataFrame:
    """
    Deduplicates the dataset grouped by sequence, resolving conflicts by quantitative
    measurements primarily, and qualitative measurements secondarily.
    """
    dedup_df: pd.DataFrame = input_df.copy()
    dedup_df['Qualitative_Mapped'] = dedup_df['Qualitative measurement'].map(QUALITATIVE_MAPPING)
    
    quantitative_subset_df: pd.DataFrame = dedup_df[dedup_df['Quantitative measurement'].notna()].copy()
    qualitative_subset_df: pd.DataFrame = dedup_df[dedup_df['Quantitative measurement'].isna()].copy()
    
    quant_seqs: set[str] = set(quantitative_subset_df['Sequence'])
    assert len(quantitative_subset_df) + len(qualitative_subset_df) == len(dedup_df), "Splits are not exhaustive"

    print("Proceeding with Deduplication")
    
    # 1. Quantitative Processing
    quantitative_consensus_df, quant_stats_df = process_quantitative_split(quantitative_subset_df)
    
    print("\n--- Merged Quantitative Calculation ---")
    print(f"Total unique sequences with quantitative data: {len(quantitative_consensus_df)}")
    
    if not quant_stats_df.empty:
        print("\nAlignment Combinations Percentages (Quant Consensus vs. Original Qual Mapped):")
        quant_crosstab = pd.crosstab(quant_stats_df['isPositive'], quant_stats_df['Qualitative_Mapped'], normalize='all') * 100
        quant_crosstab = quant_crosstab.rename_axis('Merged quantitative', axis=0).rename_axis('Mapped qualitative', axis=1)
        
        formatted_crosstab = quant_crosstab.apply(lambda x: x.map(lambda y: f"{y:.2f}"))
        print(formatted_crosstab.to_string())
    
    # 2. Qualitative Processing
    qualitative_consensus_df, cons_pos, cons_neg, conflict = process_qualitative_split(qualitative_subset_df)
    
    # Filter out qualitative results that already got a better quantitative consensus
    qualitative_only_consensus_df: pd.DataFrame = qualitative_consensus_df[~qualitative_consensus_df['Sequence'].isin(quant_seqs)].copy()
    
    total_qual_processed = cons_pos + cons_neg + conflict
    
    print("\n--- Qualitative Consensus Calculation (for sequences without quantitative data) ---")
    print(f"Total unique sequences processed for qualitative consensus: {total_qual_processed}")
    if total_qual_processed > 0:
        print(f"Sequences with consistent positive qualitative consensus: {cons_pos} ({(cons_pos/total_qual_processed)*100:.2f}%)")
        print(f"Sequences with consistent negative qualitative consensus: {cons_neg} ({(cons_neg/total_qual_processed)*100:.2f}%)")
        print(f"Sequences with conflicting (discarded): {conflict} ({(conflict/total_qual_processed)*100:.2f}%)")

    # 3. Final Merge
    final_deduplicated_df: pd.DataFrame = pd.concat([quantitative_consensus_df, qualitative_only_consensus_df], ignore_index=True)
    assert final_deduplicated_df['Sequence'].is_unique, "Deduplication failed; sequences are not unique"

    print(f"\nFinal deduplicated DataFrame has {len(final_deduplicated_df)} unique entries.\n")
    return final_deduplicated_df

Globals.deduplicated_df = deduplicate_dataset(Globals.filtered_df)


Proceeding with Deduplication



--- Merged Quantitative Calculation ---
Total unique sequences with quantitative data: 2004

Alignment Combinations Percentages (Quant Consensus vs. Original Qual Mapped):
Mapped qualitative       0      1
Merged quantitative              
0.0                  62.35  17.38
1.0                   0.00  20.27



--- Qualitative Consensus Calculation (for sequences without quantitative data) ---
Total unique sequences processed for qualitative consensus: 20717
Sequences with consistent positive qualitative consensus: 19804 (95.59%)
Sequences with consistent negative qualitative consensus: 908 (4.38%)
Sequences with conflicting (discarded): 5 (0.02%)

Final deduplicated DataFrame has 22136 unique entries.



**Conclusions on Deduplication:**
The deduplication phase successfully isolates a single ground-truth classification per unique sequence.
- **Quantitative Consistency**: The sequences condensed via quantitative averaging still retain high fidelity string alignment with their original subjective labels.
- **Qualitative Consistency**: Most non-quantitative sequences enjoy perfectly consistent qualitative labeling across assays. The rare subset of strictly qualitative entries that have directly contradictory tags (Positive in one assay, Negative in another) are safely discarded to preserve model training integrity.

## 5. Balancing Dataset with Synthetic Negatives

Our dataset is often skewed towards positive binding examples. To prevent model bias, we generate synthetic negative sequences to balance the classes. We sample the lengths from the positive distribution and amino acids from general abundance frequencies in Swiss-Prot.

**Why Random Synthesis Works:** Because the sequence space of random peptides that bind to a specific MHC complex is infinitesimally small, a randomly generated peptide is statistically guaranteed (with over >99.9% probability) to be a negative (non-binder). Thus, injecting these synthetic random sequences reliably shores up our negative counter-examples.

In [8]:
def generate_synthetic_sequence(length: int, amino_acids: Sequence[str], probabilities: Sequence[float], existing_set: Set[str]) -> str:
    """
    Generates a single valid synthetic sequence that doesn't already exist in the dataset.
    """
    while True:
        new_seq: str = ''.join(random.choices(amino_acids, weights=probabilities, k=length))
        if new_seq not in existing_set:
            return new_seq

def balance_dataset(input_df: pd.DataFrame) -> pd.DataFrame:
    """
    Generates synthetic negative sequences to balance the dataset parity.
    """
    # Amino acid frequencies from Swiss-Prot (https://web.expasy.org/protscale/pscale/A.A.Swiss-Prot.html)
    aa_probs: Mapping[str, float] = types.MappingProxyType({
        'A': 0.0825, 'R': 0.0553, 'N': 0.0406, 'D': 0.0545, 'C': 0.0137,
        'Q': 0.0393, 'E': 0.0675, 'G': 0.0707, 'H': 0.0227, 'I': 0.0596,
        'L': 0.0966, 'K': 0.0584, 'M': 0.0242, 'F': 0.0386, 'P': 0.0470,
        'S': 0.0656, 'T': 0.0534, 'W': 0.0108, 'Y': 0.0292, 'V': 0.0687
    })

    current_pos_count: int = input_df[input_df['isPositive'] == 1.0].shape[0]
    current_neg_count: int = input_df[input_df['isPositive'] == 0.0].shape[0]
    needed_negatives: int = current_pos_count - current_neg_count

    if needed_negatives <= 0:
        print("Dataset is already balanced or has more negatives.")
        return input_df
        
    print(f"Generating {needed_negatives} synthetic negative sequences to balance the dataset...")

    positive_lengths: pd.Series = input_df[input_df['isPositive'] == 1.0]['Sequence'].apply(len)
    length_counts: pd.Series = positive_lengths.value_counts(normalize=True)
    lengths: Sequence[int] = length_counts.index.tolist()
    length_probabilities: Sequence[float] = length_counts.values.tolist()

    amino_acids: Sequence[str] = list(aa_probs.keys())
    amino_acid_probabilities: Sequence[float] = list(aa_probs.values())

    all_existing_sequences_set: set[str] = set(input_df['Sequence'].tolist())
    synthetic_sequences: list[str] = []

    for _ in range(needed_negatives):
        length: int = np.random.choice(lengths, p=length_probabilities)
        new_seq: str = generate_synthetic_sequence(length, amino_acids, amino_acid_probabilities, all_existing_sequences_set)
        synthetic_sequences.append(new_seq)
        all_existing_sequences_set.add(new_seq)

    synthetic_negatives_df: pd.DataFrame = pd.DataFrame({
        'Sequence': synthetic_sequences,
        'isPositive': 0.0
    })
    
    balanced_df: pd.DataFrame = pd.concat([input_df, synthetic_negatives_df], ignore_index=True)
    return balanced_df

Globals.balanced_df = balance_dataset(Globals.deduplicated_df)


Generating 18002 synthetic negative sequences to balance the dataset...


## 6. Exporting Processed Data

Finally, we rename our working columns to the expected downstream contract (`sequence`, `is_binded`) and save the dataframe as a JSON list of records for the model training pipeline.

In [9]:
def save_processed_data(input_df: pd.DataFrame, file_path: str) -> None:
    """
    Transforms tracking column names and exports the data to JSON.
    """
    processed_df: pd.DataFrame = input_df.copy()
    processed_df.rename(columns={'isPositive': 'is_binded', 'Sequence': 'sequence'}, inplace=True)
    processed_df.to_json(file_path, orient='records', indent=2)
    print(f"Successfully saved processed dataset to {file_path}")
    print(f"Final dataset size: {len(processed_df)}")

save_processed_data(Globals.balanced_df, Globals.PROCESSED_DATA_PATH)

Successfully saved processed dataset to data/processed/training_data.json
Final dataset size: 40138
